# SQL Analysis

## Objective

Use SQL to analyze hospital characteristics, overall hospital ratings,
emergency-department performance, and differences between Washington
hospitals and national patterns using the validated SQLite database.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/hospital_quality.db")

In [2]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql_query(query, conn)

,name
0,hospitals
1,measures
2,hospital_measures


In [3]:
query = """
SELECT
    state,
    COUNT(*) AS hospital_count
FROM hospitals
GROUP BY state
ORDER BY hospital_count DESC;
"""

hospital_counts = pd.read_sql_query(query, conn)
hospital_counts

,state,hospital_count
0,TX,467
1,CA,377
2,FL,221
3,OH,193
4,IL,193
5,NY,192
6,PA,186
7,LA,161
8,MI,149
9,GA,149


### Hospital Distribution by State

We first examined how hospitals in the CMS dataset are distributed across states and territories.

The `hospitals` table was grouped by `state`, changing the analytical grain from one row per hospital to one row per state or territory.

**Key observations:**
- Texas has the largest number of hospitals in the dataset with 467.
- California follows with 377 hospitals, and Florida with 221.
- Washington has 100 hospitals.
- The dataset contains hospitals across 56 state and territory codes.

These counts describe the distribution of hospitals represented in the CMS dataset and should not be interpreted as measures of hospital quality.

In [4]:
pd.read_sql_query("PRAGMA table_info(hospitals);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,facility_id,TEXT,0,None,1
1,1,facility_name,TEXT,1,None,0
2,2,address,TEXT,0,None,0
3,3,city,TEXT,0,None,0
4,4,state,TEXT,0,None,0
5,5,zip_code,TEXT,0,None,0
6,6,county,TEXT,0,None,0
7,7,hospital_type,TEXT,0,None,0
8,8,hospital_ownership,TEXT,0,None,0
9,9,emergency_services,TEXT,0,None,0


In [5]:
query = """
SELECT
    hospital_type,
    COUNT(*) AS hospital_count
FROM hospitals
GROUP BY hospital_type
ORDER BY hospital_count DESC;
"""

hospital_types = pd.read_sql_query(query, conn)
hospital_types

,hospital_type,hospital_count
0,Acute Care Hospitals,3107
1,Critical Access Hospitals,1382
2,Psychiatric,624
3,Acute Care - Veterans Administration,132
4,Childrens,94
5,Rural Emergency Hospital,43
6,Acute Care - Department of Defense,32
7,Long-term,5


In [6]:
query = """
SELECT
    hospital_ownership,
    COUNT(*) AS hospital_count
FROM hospitals
GROUP BY hospital_ownership
ORDER BY hospital_count DESC;
"""

hospital_ownership = pd.read_sql_query(query, conn)
hospital_ownership

,hospital_ownership,hospital_count
0,Voluntary non-profit - Private,2322
1,Proprietary,1063
2,Government - Hospital District or Authority,512
3,Government - Local,393
4,Voluntary non-profit - Other,353
5,Voluntary non-profit - Church,264
6,Government - State,209
7,Veterans Health Administration,132
8,Physician,80
9,Government - Federal,42


In [7]:
query = """
SELECT
    emergency_services,
    COUNT(*) AS hospital_count
FROM hospitals
GROUP BY emergency_services
ORDER BY hospital_count DESC;
"""

emergency_services = pd.read_sql_query(query, conn)
emergency_services

,emergency_services,hospital_count
0,Yes,4495
1,No,924


### Hospital Characteristics

The CMS dataset contains 5,419 hospitals across several hospital types and ownership structures.

**Hospital type**
- Acute Care Hospitals are the largest category with 3,107 hospitals.
- Critical Access Hospitals are the second largest with 1,382.
- The dataset also includes psychiatric, Veterans Administration, children's, rural emergency, Department of Defense, and long-term hospitals.

**Hospital ownership**
- Voluntary non-profit private hospitals are the largest ownership category with 2,322 hospitals.
- Proprietary hospitals are the second largest with 1,063.
- The remaining hospitals span government, nonprofit, Veterans Health Administration, physician, Department of Defense, and tribal ownership categories.

**Emergency services**
- 4,495 hospitals report providing emergency services.
- 924 report that they do not.

These results describe the composition of hospitals represented in the CMS dataset. They establish the hospital population that will be used in subsequent quality and performance analyses.

In [8]:
query = """
SELECT
    overall_rating_numeric AS rating,
    COUNT(*) AS hospital_count
FROM hospitals
WHERE overall_rating_numeric IS NOT NULL
GROUP BY overall_rating_numeric
ORDER BY overall_rating_numeric;
"""

rating_distribution = pd.read_sql_query(query, conn)
rating_distribution

,rating,hospital_count
0,1.0,198
1,2.0,661
2,3.0,985
3,4.0,946
4,5.0,384


In [9]:
query = """
SELECT
    COUNT(*) AS total_hospitals,
    COUNT(overall_rating_numeric) AS rated_hospitals,
    COUNT(*) - COUNT(overall_rating_numeric) AS unrated_hospitals,
    ROUND(
        100.0 * COUNT(overall_rating_numeric) / COUNT(*),
        1
    ) AS percent_rated
FROM hospitals;
"""

rating_coverage = pd.read_sql_query(query, conn)
rating_coverage

,total_hospitals,rated_hospitals,unrated_hospitals,percent_rated
0,5419,3174,2245,58.6


In [10]:
query = """
SELECT
    ROUND(AVG(overall_rating_numeric), 2) AS average_rating,
    MIN(overall_rating_numeric) AS minimum_rating,
    MAX(overall_rating_numeric) AS maximum_rating
FROM hospitals
WHERE overall_rating_numeric IS NOT NULL;
"""

rating_summary = pd.read_sql_query(query, conn)
rating_summary

,average_rating,minimum_rating,maximum_rating
0,3.21,1.0,5.0


### Overall Hospital Ratings

Of the 5,419 hospitals in the dataset, 3,174 (58.6%) have a numeric overall hospital rating, while 2,245 do not. Therefore, analyses involving overall ratings represent the rated subset of hospitals rather than the full hospital population.

**Rating distribution**
- 1 star: 198 hospitals
- 2 stars: 661 hospitals
- 3 stars: 985 hospitals
- 4 stars: 946 hospitals
- 5 stars: 384 hospitals

Among hospitals with a numeric rating:
- The average overall rating is 3.21.
- Ratings range from 1 to 5.
- Three- and four-star hospitals are the most common rating categories.

Because 41.4% of hospitals do not have a numeric overall rating, comparisons involving ratings should account for this incomplete coverage.

In [11]:
query = """
SELECT
    hospital_type,
    COUNT(overall_rating_numeric) AS rated_hospitals,
    ROUND(AVG(overall_rating_numeric), 2) AS average_rating
FROM hospitals
WHERE overall_rating_numeric IS NOT NULL
GROUP BY hospital_type
ORDER BY average_rating DESC;
"""

ratings_by_type = pd.read_sql_query(query, conn)
ratings_by_type

,hospital_type,rated_hospitals,average_rating
0,Acute Care - Veterans Administration,112,4.16
1,Critical Access Hospitals,399,3.22
2,Acute Care Hospitals,2663,3.16


In [12]:
query = """
SELECT
    hospital_type,
    COUNT(*) AS total_hospitals,
    COUNT(overall_rating_numeric) AS rated_hospitals,
    ROUND(
        100.0 * COUNT(overall_rating_numeric) / COUNT(*),
        1
    ) AS percent_rated
FROM hospitals
GROUP BY hospital_type
ORDER BY percent_rated DESC;
"""

rating_coverage_by_type = pd.read_sql_query(query, conn)
rating_coverage_by_type

,hospital_type,total_hospitals,rated_hospitals,percent_rated
0,Acute Care Hospitals,3107,2663,85.7
1,Acute Care - Veterans Administration,132,112,84.8
2,Critical Access Hospitals,1382,399,28.9
3,Rural Emergency Hospital,43,0,0.0
4,Psychiatric,624,0,0.0
5,Long-term,5,0,0.0
6,Childrens,94,0,0.0
7,Acute Care - Department of Defense,32,0,0.0


### Overall Ratings by Hospital Type

Overall rating availability varies substantially by hospital type.

- Acute Care Hospitals have the highest rating coverage: 2,663 of 3,107 hospitals (85.7%) have numeric ratings, with an average rating of 3.16.
- Acute Care - Veterans Administration hospitals have similar coverage at 84.8% (112 of 132) and an average rating of 4.16.
- Critical Access Hospitals have much lower rating coverage: only 399 of 1,382 hospitals (28.9%) have numeric ratings, with an average rating of 3.22.
- The remaining five hospital types have no numeric overall ratings in this dataset.

The higher average rating observed among Veterans Administration hospitals is descriptive and should not by itself be interpreted as evidence that hospital type causes higher ratings. Differences in rating eligibility, hospital characteristics, patient populations, or CMS methodology may contribute to the observed pattern.

Because rating coverage differs substantially across hospital types, comparisons of average ratings should be interpreted alongside the number and percentage of hospitals that actually receive ratings.

In [13]:
query = """
SELECT *
FROM measures
ORDER BY measure_id;
"""

measures_list = pd.read_sql_query(query, conn)
measures_list

,measure_id,measure_name,condition
0,EDV,Emergency department volume,Emergency Department
1,GMCS,Global Malnutrition Composite Score,Electronic Clinical Quality Measure
2,GMCS_Malnutrition_Diagnosis_Documented,Global Malnutrition Composite Score: Malnutrit...,Electronic Clinical Quality Measure
3,GMCS_Malnutrition_Screening,Global Malnutrition Composite Score: Malnutrit...,Electronic Clinical Quality Measure
4,GMCS_Nutrition_Assessment,Global Malnutrition Composite Score: Nutrition...,Electronic Clinical Quality Measure
5,GMCS_Nutritional_Care_Plan,Global Malnutrition Composite Score: Nutrition...,Electronic Clinical Quality Measure
6,HH_HYPER,Hospital Harm - Severe Hyperglycemia,Electronic Clinical Quality Measure
7,HH_HYPO,Hospital Harm - Severe Hypoglycemia,Electronic Clinical Quality Measure
8,HH_ORAE,Hospital Harm - Opioid Related Adverse Events,Electronic Clinical Quality Measure
9,IMM_3,Healthcare workers given influenza vaccination,Healthcare Personnel Vaccination


In [14]:
query = """
SELECT
    m.measure_id,
    m.measure_name,
    COUNT(hm.facility_id) AS hospital_records,
    COUNT(hm.score_numeric) AS numeric_scores
FROM measures AS m
JOIN hospital_measures AS hm
    ON m.measure_id = hm.measure_id
WHERE m.condition = 'Emergency Department'
GROUP BY
    m.measure_id,
    m.measure_name
ORDER BY m.measure_id;
"""

ed_measure_coverage = pd.read_sql_query(query, conn)
ed_measure_coverage

,measure_id,measure_name,hospital_records,numeric_scores
0,EDV,Emergency department volume,4658,0
1,OP_18a,Average (median) time all patients spent in th...,4658,4052
2,OP_18b,Average (median) time patients spent in the em...,4658,4081
3,OP_18c,Average (median) time psychiatric/mental healt...,4658,3023
4,OP_18d,Average (median) time patients spent in the em...,4658,2364
5,OP_22,Left before being seen,4658,3821
6,OP_23,Head CT results,4658,1492


In [17]:
for _, row in pd.read_sql_query("""
SELECT measure_id, measure_name
FROM measures
WHERE condition = 'Emergency Department'
ORDER BY measure_id;
""", conn).iterrows():
    print(f"{row['measure_id']}: {row['measure_name']}")

EDV: Emergency department volume
OP_18a: Average (median) time all patients spent in the emergency department before leaving from the visit, including psychiatric/mental health patients and patients who were transferred to another facility. A lower number of minutes is better
OP_18b: Average (median) time patients spent in the emergency department before leaving from the visit, excluding patients transferred to another facility or psychiatric care/mental health patients. A lower number of minutes is better
OP_18c: Average (median) time psychiatric/mental health patients spent in the emergency department before leaving from the visit. A lower number of minutes is better
OP_18d: Average (median) time patients spent in the emergency department before being transferred to another facility. A lower number of minutes is better
OP_22: Left before being seen
OP_23: Head CT results


In [18]:
pd.set_option("display.max_colwidth", None)

In [19]:
query = """
SELECT
    COUNT(score_numeric) AS hospitals_with_scores,
    ROUND(AVG(score_numeric), 1) AS average_minutes,
    MIN(score_numeric) AS minimum_minutes,
    MAX(score_numeric) AS maximum_minutes
FROM hospital_measures
WHERE measure_id = 'OP_18b'
  AND score_numeric IS NOT NULL;
"""

op18b_summary = pd.read_sql_query(query, conn)
op18b_summary

,hospitals_with_scores,average_minutes,minimum_minutes,maximum_minutes
0,4081,156.9,42.0,413.0


### Emergency Department Performance

Seven measures in the dataset are classified under the Emergency Department condition. Numeric score availability varies across these measures.

OP_18b was selected as the primary measure for analyzing emergency-department performance. It measures the median time patients spent in the emergency department before leaving, excluding patients transferred to another facility and psychiatric/mental-health patients. Lower values indicate better performance.

OP_18b has numeric scores for 4,081 hospitals, the highest numeric coverage among the emergency-department measures in the dataset.

Across hospitals with numeric OP_18b scores:
- The average hospital-reported median ED time is 156.9 minutes.
- Hospital-level median times range from 42 to 413 minutes.

The 156.9-minute value is an average of hospital-level median times and should not be interpreted as the national patient-level median ED time.

In [20]:
query = """
SELECT
    CASE
        WHEN h.state = 'WA' THEN 'Washington'
        ELSE 'Other U.S.'
    END AS geography,
    COUNT(hm.score_numeric) AS hospitals_with_scores,
    ROUND(AVG(hm.score_numeric), 1) AS average_minutes
FROM hospital_measures AS hm
JOIN hospitals AS h
    ON hm.facility_id = h.facility_id
WHERE hm.measure_id = 'OP_18b'
  AND hm.score_numeric IS NOT NULL
GROUP BY geography;
"""

op18b_wa_comparison = pd.read_sql_query(query, conn)
op18b_wa_comparison

,geography,hospitals_with_scores,average_minutes
0,Other U.S.,4006,156.7
1,Washington,75,169.5


In [21]:
query = """
SELECT
    h.hospital_type,
    CASE
        WHEN h.state = 'WA' THEN 'Washington'
        ELSE 'Other U.S.'
    END AS geography,
    COUNT(hm.score_numeric) AS hospitals_with_scores,
    ROUND(AVG(hm.score_numeric), 1) AS average_minutes
FROM hospital_measures AS hm
JOIN hospitals AS h
    ON hm.facility_id = h.facility_id
WHERE hm.measure_id = 'OP_18b'
  AND hm.score_numeric IS NOT NULL
GROUP BY
    h.hospital_type,
    geography
ORDER BY
    h.hospital_type,
    geography;
"""

op18b_by_type = pd.read_sql_query(query, conn)
op18b_by_type

,hospital_type,geography,hospitals_with_scores,average_minutes
0,Acute Care - Department of Defense,Other U.S.,30,142.5
1,Acute Care - Department of Defense,Washington,1,179.0
2,Acute Care Hospitals,Other U.S.,2752,175.0
3,Acute Care Hospitals,Washington,45,194.4
4,Critical Access Hospitals,Other U.S.,1224,115.9
5,Critical Access Hospitals,Washington,29,130.4


In [22]:
query = """
SELECT
    CASE
        WHEN h.state = 'WA' THEN 'Washington'
        ELSE 'Other U.S.'
    END AS geography,
    h.hospital_type,
    COUNT(*) AS hospitals_with_scores,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (
            PARTITION BY
                CASE
                    WHEN h.state = 'WA' THEN 'Washington'
                    ELSE 'Other U.S.'
                END
        ),
        1
    ) AS percent_of_group
FROM hospital_measures AS hm
JOIN hospitals AS h
    ON hm.facility_id = h.facility_id
WHERE hm.measure_id = 'OP_18b'
  AND hm.score_numeric IS NOT NULL
GROUP BY
    geography,
    h.hospital_type
ORDER BY
    geography,
    hospitals_with_scores DESC;
"""

op18b_hospital_mix = pd.read_sql_query(query, conn)
op18b_hospital_mix

,geography,hospital_type,hospitals_with_scores,percent_of_group
0,Other U.S.,Acute Care Hospitals,2752,68.7
1,Other U.S.,Critical Access Hospitals,1224,30.6
2,Other U.S.,Acute Care - Department of Defense,30,0.7
3,Washington,Acute Care Hospitals,45,60.0
4,Washington,Critical Access Hospitals,29,38.7
5,Washington,Acute Care - Department of Defense,1,1.3


### Washington vs. Other U.S. Hospitals

Washington hospitals with numeric OP_18b scores have an average hospital-level median emergency-department time of 169.5 minutes, compared with 156.7 minutes among hospitals outside Washington. This represents a descriptive difference of 12.8 minutes, with lower OP_18b values indicating better performance.

Because hospital type may affect emergency-department performance, the comparison was also examined separately by hospital type.

Among Acute Care Hospitals, Washington hospitals average 194.4 minutes compared with 175.0 minutes outside Washington. Among Critical Access Hospitals, Washington hospitals average 130.4 minutes compared with 115.9 minutes outside Washington. The Department of Defense category contains only one Washington hospital and is therefore not useful for a group-level comparison.

Hospital mix does not appear to explain the overall Washington difference. Critical Access Hospitals, which have shorter OP_18b times in both geographic groups, make up 38.7% of Washington hospitals with OP_18b scores compared with 30.6% outside Washington.

These results are descriptive. They do not establish that location causes differences in emergency-department performance, nor do they account for other hospital or patient characteristics that may influence OP_18b times.

In [24]:
query = """
SELECT
    h.overall_rating_numeric AS overall_rating,
    COUNT(*) AS hospitals_with_scores,
    ROUND(AVG(hm.score_numeric), 1) AS average_ed_minutes
FROM hospitals AS h
JOIN hospital_measures AS hm
    ON h.facility_id = hm.facility_id
WHERE hm.measure_id = 'OP_18b'
  AND hm.score_numeric IS NOT NULL
  AND h.overall_rating_numeric IS NOT NULL
GROUP BY h.overall_rating_numeric
ORDER BY h.overall_rating_numeric;
"""

rating_vs_ed = pd.read_sql_query(query, conn)
rating_vs_ed

,overall_rating,hospitals_with_scores,average_ed_minutes
0,1.0,187,178.4
1,2.0,638,174.1
2,3.0,945,173.5
3,4.0,877,168.0
4,5.0,319,175.1


In [25]:
query = """
SELECT
    h.overall_rating_numeric AS overall_rating,
    COUNT(*) AS hospitals_with_scores,
    ROUND(AVG(hm.score_numeric), 1) AS average_ed_minutes
FROM hospitals AS h
JOIN hospital_measures AS hm
    ON h.facility_id = hm.facility_id
WHERE hm.measure_id = 'OP_18b'
  AND hm.score_numeric IS NOT NULL
  AND h.overall_rating_numeric IS NOT NULL
  AND h.hospital_type = 'Acute Care Hospitals'
GROUP BY h.overall_rating_numeric
ORDER BY h.overall_rating_numeric;
"""

acute_rating_vs_ed = pd.read_sql_query(query, conn)
acute_rating_vs_ed

,overall_rating,hospitals_with_scores,average_ed_minutes
0,1.0,170,183.5
1,2.0,562,180.2
2,3.0,830,179.1
3,4.0,753,175.2
4,5.0,277,183.5


### Overall Hospital Rating and ED Performance

We examined whether overall hospital ratings are associated with OP_18b emergency-department times.

Across rated hospitals with numeric OP_18b scores, average ED times generally decrease from 1-star through 4-star hospitals:

- 1 star: 178.4 minutes
- 2 stars: 174.1 minutes
- 3 stars: 173.5 minutes
- 4 stars: 168.0 minutes
- 5 stars: 175.1 minutes

The pattern is not monotonic because 5-star hospitals have longer average OP_18b times than 4-star hospitals.

To reduce the influence of differences in hospital type, the analysis was repeated using only Acute Care Hospitals. A similar pattern remained: average OP_18b times declined from 183.5 minutes among 1-star hospitals to 175.2 minutes among 4-star hospitals, but increased to 183.5 minutes among 5-star hospitals.

These results suggest that overall hospital rating and OP_18b performance are related in some rating categories, but the relationship is not simply linear. Overall hospital ratings reflect broader dimensions of hospital quality, and this analysis does not establish that ED performance determines a hospital's overall rating.

In [26]:
query = """
WITH wa_ed AS (
    SELECT
        h.facility_id,
        h.facility_name,
        h.hospital_type,
        h.overall_rating_numeric,
        hm.score_numeric AS op18b_minutes
    FROM hospitals AS h
    JOIN hospital_measures AS hm
        ON h.facility_id = hm.facility_id
    WHERE h.state = 'WA'
      AND hm.measure_id = 'OP_18b'
      AND hm.score_numeric IS NOT NULL
)

SELECT
    facility_id,
    facility_name,
    hospital_type,
    overall_rating_numeric,
    op18b_minutes,
    RANK() OVER (
        PARTITION BY hospital_type
        ORDER BY op18b_minutes ASC
    ) AS rank_within_type
FROM wa_ed
ORDER BY
    hospital_type,
    rank_within_type;
"""

wa_ed_rankings = pd.read_sql_query(query, conn)
wa_ed_rankings

,facility_id,facility_name,hospital_type,overall_rating_numeric,op18b_minutes,rank_within_type
0,50005F,Madigan AMC (FT Lewis),Acute Care - Department of Defense,NaN,179.0,1
1,500079,MULTICARE GOOD SAMARITAN HOSPITAL,Acute Care Hospitals,2.0,133.0,1
2,500002,PROVIDENCE ST MARY MEDICAL CENTER,Acute Care Hospitals,3.0,134.0,2
3,500033,SAMARITAN HOSPITAL,Acute Care Hospitals,2.0,135.0,3
4,500139,CAPITAL MEDICAL CENTER,Acute Care Hospitals,3.0,140.0,4
...,...,...,...,...,...,...
70,501327,WHITMAN HOSPITAL AND MEDICAL CENTER,Critical Access Hospitals,5.0,156.0,25
71,501304,SUMMIT PACIFIC MEDICAL CENTER,Critical Access Hospitals,2.0,158.0,26
72,501335,ST ELIZABETH HOSPITAL,Critical Access Hospitals,5.0,193.0,27
73,501336,MASON GENERAL HOSPITAL & FAMILY OF CLINICS,Critical Access Hospitals,2.0,200.0,28


In [27]:
wa_ed_rankings.tail(10)

,facility_id,facility_name,hospital_type,overall_rating_numeric,op18b_minutes,rank_within_type
65,501325,FORKS COMMUNITY HOSPITAL,Critical Access Hospitals,NaN,136.0,20
66,501323,JEFFERSON HEALTHCARE,Critical Access Hospitals,3.0,139.0,21
67,501329,PEACEHEALTH UNITED GENERAL MEDICAL CENTER,Critical Access Hospitals,3.0,142.0,22
68,501332,TRI-STATE MEMORIAL HOSPITAL,Critical Access Hospitals,4.0,143.0,23
69,501326,PROVIDENCE MOUNT CARMEL HOSPITAL,Critical Access Hospitals,3.0,150.0,24
70,501327,WHITMAN HOSPITAL AND MEDICAL CENTER,Critical Access Hospitals,5.0,156.0,25
71,501304,SUMMIT PACIFIC MEDICAL CENTER,Critical Access Hospitals,2.0,158.0,26
72,501335,ST ELIZABETH HOSPITAL,Critical Access Hospitals,5.0,193.0,27
73,501336,MASON GENERAL HOSPITAL & FAMILY OF CLINICS,Critical Access Hospitals,2.0,200.0,28
74,501328,MID VALLEY HOSPITAL & CLINIC,Critical Access Hospitals,NaN,202.0,29


In [28]:
wa_ed_rankings[
    wa_ed_rankings["rank_within_type"].isna()
]

,facility_id,facility_name,hospital_type,overall_rating_numeric,op18b_minutes,rank_within_type


### Washington Hospital Benchmarking

Washington hospitals were benchmarked using OP_18b by ranking each hospital against other Washington hospitals of the same hospital type. Because lower OP_18b values represent shorter emergency-department times, lower rank numbers indicate shorter reported median ED times within each hospital-type group.

A Common Table Expression (CTE) was used to first create a Washington-specific analytical dataset containing hospital characteristics and OP_18b scores. A `RANK()` window function then ranked hospitals within each hospital type without collapsing the individual hospital records.

For example, among Washington Acute Care Hospitals with numeric OP_18b scores, MultiCare Good Samaritan Hospital has the shortest reported median time at 133 minutes. Among Washington Critical Access Hospitals, observed OP_18b times extend to 202 minutes for Mid Valley Hospital & Clinic.

These rankings are relative benchmarks within hospital type and should not be interpreted as comprehensive rankings of hospital quality. OP_18b represents one specific dimension of emergency-department performance.

In [29]:
print("Hospital rows:", pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM hospitals", conn
).iloc[0, 0])

print("Measure rows:", pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM measures", conn
).iloc[0, 0])

print("Hospital-measure rows:", pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM hospital_measures", conn
).iloc[0, 0])

print("WA OP_18b hospitals:", len(wa_ed_rankings))

Hospital rows: 5419
Measure rows: 30
Hospital-measure rows: 138084
WA OP_18b hospitals: 75


## SQL Analysis Summary

This analysis used the validated SQLite relational database to examine hospital characteristics, overall hospital ratings, emergency-department performance, and Washington hospitals relative to broader U.S. patterns.

Key analytical findings include:

- The dataset contains 5,419 hospitals across 56 state and territory codes, with Acute Care Hospitals representing the largest hospital type.
- Numeric overall hospital ratings are available for 3,174 hospitals (58.6%), with substantial differences in rating coverage across hospital types.
- OP_18b was selected as the primary emergency-department performance measure because of its clear interpretation and relatively high numeric coverage.
- Among 4,081 hospitals with numeric OP_18b scores, the average hospital-reported median ED time is 156.9 minutes.
- Washington hospitals with OP_18b scores average 169.5 minutes compared with 156.7 minutes among hospitals outside Washington.
- The Washington difference persists when Acute Care Hospitals and Critical Access Hospitals are examined separately, although these comparisons are descriptive and do not establish causation.
- Overall hospital rating and OP_18b performance show some relationship from 1-star through 4-star hospitals, but the pattern is not monotonic because 5-star hospitals have longer average OP_18b times than 4-star hospitals.
- Washington hospitals were benchmarked within hospital type using a CTE and the `RANK()` window function.

The analysis demonstrates how relational SQL can move from descriptive profiling to increasingly focused comparisons while accounting for data coverage, hospital characteristics, and appropriate limitations.